# Qualidade de `atividades.parquet`

Diagnóstico do log de treinos de 2026 pelos mesmos cinco critérios usados em `eventos_assinatura.csv` e `pagamentos.csv`:

| Critério | Pergunta |
|---|---|
| **Completude** | Está tudo lá? Campos preenchidos e treinos que deveriam existir |
| **Validade** | Os valores estão no formato e no domínio esperados? |
| **Consistência** | Os dados concordam entre si, com o cadastro e com o `uso_mensal.csv`? |
| **Unicidade** | Cada treino aparece uma vez só? |
| **Temporalidade** | As datas estão no período certo, no fuso certo, sem buracos? |

**Só diagnóstico:** nada é corrigido aqui. Cada achado vai para uma tabela-resumo em `saidas/`, com status
(`ok`, `esperado` ou `atenção`) e tratamento sugerido. Nada em `dados/brutos/` é alterado (hash conferido no início e no fim).

**Arquivos cruzados:** `usuarios.csv` (cadastro, contas de teste e relógio conectado) e `uso_mensal.csv` (o mesmo
acontecimento, treinos por mês, contado pelo BI).

**Decisões já registradas que valem aqui** (`decisoes.md`): datas em horário de Brasília antes de contar por dia ou mês;
exclusão das contas de teste (`@vitaliza.com.br`).

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
SAIDAS = ROOT / "saidas"
INICIO_ARQUIVO = pd.Timestamp("2026-01-01")  # o dicionário: treinos de 01/01 a 30/09/2026
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)    # 01/10/2026 00:00, horário de Brasília

ARQUIVOS = ["atividades.parquet", "usuarios.csv", "uso_mensal.csv"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

achados = []

def achado(criterio, checagem, resultado, linhas, tratamento, status=None):
    """status: ok (nada encontrado), esperado (encontrado, mas é comportamento do negócio) ou atenção.
    Sem status explícito: ok quando nenhuma linha é afetada, atenção caso contrário."""
    if status is None:
        status = "ok" if linhas == 0 else "atenção"
    achados.append({"criterio": criterio, "checagem": checagem, "status": status,
                    "resultado": resultado, "linhas_afetadas": int(linhas), "tratamento_sugerido": tratamento})

## 0. Leitura

O parquet guarda tipos: `duracao_min`, `distancia_km`, `ritmo_min_km` e `fc_media` chegam como número, `gps_ok` como
booleano e os vazios como nulo. As colunas de texto ficam como chegaram. Os CSVs cruzados são lidos como texto.

In [2]:
ativ = pd.read_parquet(BRUTOS / "atividades.parquet")
usuarios = pd.read_csv(BRUTOS / "usuarios.csv", dtype=str, keep_default_na=False,
                       usecols=["user_id", "email", "data_cadastro", "relogio_conectado"])
uso = pd.read_csv(BRUTOS / "uso_mensal.csv", dtype={"user_id": str, "mes": str})

assert len(ativ) == 1_254_475  # linhas segundo o dicionário
N = len(ativ)
COLUNAS = list(ativ.columns)
print(f"atividades: {N:,} linhas x {ativ.shape[1]} colunas -> {COLUNAS}")
print("tipos:", ativ.dtypes.astype(str).to_dict())
ativ.sample(5, random_state=0)

atividades: 1,254,475 linhas x 11 colunas -> ['atividade_id', 'user_id', 'inicio', 'tipo', 'duracao_min', 'distancia_km', 'ritmo_min_km', 'fc_media', 'origem', 'gps_ok', 'treino_com']
tipos: {'atividade_id': 'object', 'user_id': 'object', 'inicio': 'object', 'tipo': 'object', 'duracao_min': 'float64', 'distancia_km': 'float64', 'ritmo_min_km': 'float64', 'fc_media': 'float64', 'origem': 'object', 'gps_ok': 'bool', 'treino_com': 'object'}


,atividade_id,user_id,inicio,tipo,duracao_min,distancia_km,ritmo_min_km,fc_media,origem,gps_ok,treino_com
666416,a_0852880,u_f74790d0,2026-05-23 06:52:00,ride,51.2,20.79,2.46,NaN,app,True,
1143428,a_1027391,u_1d9dbbca,2026-09-09 06:41:00,strength,37.9,NaN,NaN,NaN,app,True,
682324,a_0586642,u_f5cf326c,2026-05-26 18:07:00,hiit,24.0,NaN,NaN,NaN,app,True,
790058,a_1253389,u_592a0a0d,2026-06-20 08:11:00,yoga,23.1,NaN,NaN,NaN,app,True,u_5fe1679c
764432,a_0717654,u_a42b6108,2026-06-14 07:47:00,run,45.3,9.36,4.85,NaN,app,True,


### Colunas derivadas para a análise

- `inicio_txt`: o `inicio` como data, sem mexer no fuso;
- `dh_br`, `data_br`, `mes_br`: início em horário de Brasília. **Os treinos do Garmin chegam em UTC sem o "Z"** e
  recebem −3h; os demais já estão em Brasília. A hipótese é testada na seção 3.1;
- `tipo_pt`: o tipo com os cinco nomes em inglês trocados pelo equivalente em português (run → corrida, walk → caminhada,
  ride → pedal, strength → forca, hiit → funcional). A equivalência é testada na seção 5.4;
- `ar_livre`: corrida, caminhada ou pedal (os treinos com GPS); os demais são vídeo;
- `relogio`, `cadastro`: relógio conectado e data de cadastro da conta, de `usuarios.csv`.

O arquivo original não muda.

In [3]:
TIPO_PT = {"run": "corrida", "walk": "caminhada", "ride": "pedal", "strength": "forca", "hiit": "funcional"}
AR_LIVRE = {"corrida", "caminhada", "pedal"}

ativ["inicio_txt"] = pd.to_datetime(ativ["inicio"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
eh_garmin = ativ["origem"] == "garmin"
ativ["dh_br"] = ativ["inicio_txt"].where(~eh_garmin, ativ["inicio_txt"] - pd.Timedelta(hours=3))
ativ["data_br"] = ativ["dh_br"].dt.normalize()
ativ["mes_br"] = ativ["dh_br"].dt.strftime("%Y-%m")
ativ["tipo_pt"] = ativ["tipo"].replace(TIPO_PT)
ativ["ar_livre"] = ativ["tipo_pt"].isin(AR_LIVRE)

info = usuarios.set_index("user_id")
ativ["relogio"] = ativ["user_id"].map(info["relogio_conectado"])
ativ["cadastro"] = pd.to_datetime(ativ["user_id"].map(info["data_cadastro"]), format="%Y-%m-%d %H:%M:%S")

ativ = ativ.sort_values(["user_id", "dh_br", "atividade_id"], kind="stable").reset_index(drop=True)
eh_garmin = ativ["origem"] == "garmin"
ativ[["atividade_id", "origem", "inicio", "dh_br", "tipo", "tipo_pt", "relogio"]].head()

,atividade_id,origem,inicio,dh_br,tipo,tipo_pt,relogio
0,a_1047734,app,2026-01-03 07:00:00,2026-01-03 07:00:00,funcional,funcional,nenhum
1,a_0738337,app,2026-01-03 07:22:00,2026-01-03 07:22:00,funcional,funcional,nenhum
2,a_1196588,app,2026-01-03 09:07:00,2026-01-03 09:07:00,funcional,funcional,nenhum
3,a_0071018,app,2026-01-05 18:11:00,2026-01-05 18:11:00,caminhada,caminhada,nenhum
4,a_0595444,app,2026-01-05 18:56:00,2026-01-05 18:56:00,forca,forca,nenhum


## 1. Completude

### 1.1 Campos preenchidos

Todo treino deve ter id, conta, início, tipo, duração, origem e `gps_ok`. Distância e ritmo só existem com GPS,
frequência cardíaca só com relógio e `treino_com` só quando a pessoa marca companhia: nessas, vazio é esperado e é
cruzado com a categoria nas próximas células.

In [4]:
VAZIOS_DISFARCADOS = {"nan", "null", "none", "na", "n/a", "-"}

def vazio(s):
    return s.isna() | (s.astype(str).str.strip() == "") if s.dtype == object else s.isna()

def disfarcado(s):
    return s.notna() & s.astype(str).str.strip().str.lower().isin(VAZIOS_DISFARCADOS) if s.dtype == object \
        else pd.Series(False, index=s.index)

vazios = pd.DataFrame({"vazio": [vazio(ativ[c]).sum() for c in COLUNAS],
                       "vazio_disfarcado": [disfarcado(ativ[c]).sum() for c in COLUNAS]}, index=COLUNAS)
vazios["% vazio"] = (100 * vazios["vazio"] / N).round(1)
display(vazios)

OBRIGATORIAS = ["atividade_id", "user_id", "inicio", "tipo", "duracao_min", "origem", "gps_ok"]
faltando = int(vazios.loc[OBRIGATORIAS, ["vazio", "vazio_disfarcado"]].values.sum())
achado("Completude", "vazio nas colunas que todo treino tem", f"{faltando:,} células em {', '.join(OBRIGATORIAS)}",
       faltando, "Nenhum")

,vazio,vazio_disfarcado,% vazio
atividade_id,0,0,0.0
user_id,0,0,0.0
inicio,0,0,0.0
tipo,0,0,0.0
duracao_min,0,0,0.0
distancia_km,574835,0,45.8
ritmo_min_km,582290,0,46.4
fc_media,764987,0,61.0
origem,0,0,0.0
gps_ok,0,0,0.0


### 1.2 Distância e ritmo × tipo de treino

Espera-se distância e ritmo em todo treino ao ar livre e em nenhum treino de vídeo.

In [5]:
por_tipo = ativ.groupby("tipo_pt").agg(
    treinos=("atividade_id", "size"),
    com_distancia=("distancia_km", lambda s: s.notna().sum()),
    com_ritmo=("ritmo_min_km", lambda s: s.notna().sum()),
    gps_falhou=("gps_ok", lambda s: (~s).sum()))
display(por_tipo)

video_com_gps = int((~ativ["ar_livre"] & (ativ["distancia_km"].notna() | ativ["ritmo_min_km"].notna())).sum())
ar_sem_dist = ativ["ar_livre"] & ativ["distancia_km"].isna()
ar_sem_ritmo = ativ["ar_livre"] & ativ["distancia_km"].notna() & ativ["ritmo_min_km"].isna()
faltam = ar_sem_dist | ar_sem_ritmo
dist_zero = int((ativ["distancia_km"] == 0).sum())
print(f"vídeo com distância ou ritmo: {video_com_gps:,}")
print(f"ao ar livre sem distância: {ar_sem_dist.sum():,} | com distância e sem ritmo: {ar_sem_ritmo.sum():,} "
      f"| distância igual a zero: {dist_zero:,}")
print("gps_ok nesses treinos:", ativ.loc[faltam, "gps_ok"].value_counts().to_dict(),
      "| gps_ok = False fora deles:", int((~ativ["gps_ok"] & ~faltam).sum()))
print("origem:", ativ.loc[faltam, "origem"].value_counts().to_dict(), "| tipo:", ativ.loc[faltam, "tipo"].value_counts().to_dict())

achado("Completude", "treino de vídeo com distância ou ritmo", f"{video_com_gps:,} linhas", video_com_gps, "Nenhum")
achado("Completude", "treino ao ar livre sem distância ou sem ritmo",
       f"{ar_sem_dist.sum():,} sem distância e {ar_sem_ritmo.sum():,} com distância e sem ritmo ({dist_zero:,} com distância 0); "
       "todos com gps_ok = False, todos do app: é a falha de GPS de maio e junho (seção 5.3)",
       faltam.sum(),
       "Não imputar distância nem ritmo: marcar o treino como 'GPS falhou' e usá-lo só na contagem e na duração. "
       "A falha em si é variável candidata para o churn (problema técnico vivido pela pessoa)")

,treinos,com_distancia,com_ritmo,gps_falhou
tipo_pt,,,,
caminhada,206533,204332,202109,4424
corrida,385926,381902,377691,8235
forca,198252,0,0,0
funcional,255410,0,0,0
pedal,94505,93406,92385,2120
yoga,113849,0,0,0


vídeo com distância ou ritmo: 0
ao ar livre sem distância: 7,324 | com distância e sem ritmo: 7,455 | distância igual a zero: 23
gps_ok nesses treinos: {False: 14779} | gps_ok = False fora deles: 0
origem: {'app': 14779} | tipo: {'run': 8235, 'walk': 4424, 'ride': 2120}


### 1.3 Contas de teste

In [6]:
teste = set(usuarios.loc[usuarios["email"].str.strip().str.lower().str.endswith("@vitaliza.com.br"), "user_id"])
ativ_teste = int(ativ["user_id"].isin(teste).sum())
companhia_teste = int(ativ["treino_com"].isin(teste).sum())
print(f"treinos de contas de teste: {ativ_teste:,} | treinos com conta de teste como companhia: {companhia_teste:,}")
achado("Completude", "treinos de contas de teste", f"{ativ_teste:,} como autor e {companhia_teste:,} como companhia",
       ativ_teste + companhia_teste, "Se houver: excluir, como em assinaturas.csv")

treinos de contas de teste: 0 | treinos com conta de teste como companhia: 0


### 1.4 Treinos que deveriam estar no `uso_mensal.csv`, nos dois sentidos

Em 2026 o `uso_mensal.csv` tem uma linha para cada conta e mês, inclusive com zero treinos. Então todo usuário-mês com
treino no parquet deveria ter linha lá com `atividades` > 0, e todo usuário-mês com `atividades` > 0 lá deveria ter
treino aqui. As quantidades são comparadas na seção 3.6.

In [7]:
n_parquet = ativ.groupby(["user_id", "mes_br"]).size().rename("treinos_parquet")
n_parquet.index.names = ["user_id", "mes"]
uso26 = uso[uso["mes"] >= "2026-01"].set_index(["user_id", "mes"])

sem_linha = n_parquet.index.difference(uso26.index)
com_zero = uso26.index[uso26["atividades"] == 0].intersection(n_parquet.index)
uso_sem_parquet = uso26.index[uso26["atividades"] > 0].difference(n_parquet.index)
contas_sem_linha = sem_linha.get_level_values("user_id").unique()
print(f"usuário-mês com treino no parquet: {len(n_parquet):,}")
print(f"  sem linha no uso_mensal: {len(sem_linha):,} ({len(contas_sem_linha):,} contas; "
      f"{(~contas_sem_linha.isin(uso26.index.get_level_values('user_id'))).sum():,} delas sem nenhuma linha em 2026)")
print(f"  com linha no uso_mensal e atividades = 0: {len(com_zero):,}")
print(f"usuário-mês com atividades > 0 no uso_mensal e nenhum treino no parquet: {len(uso_sem_parquet):,}")

achado("Completude", "usuário-mês com treino no parquet e sem linha no uso_mensal",
       f"{len(sem_linha):,} usuário-mês de {len(contas_sem_linha):,} contas; outros {len(com_zero):,} têm linha com atividades = 0",
       len(sem_linha), "Para medir uso em 2026, partir do parquet, não do uso_mensal (ver 3.6)")
achado("Completude", "usuário-mês com treino no uso_mensal e nenhum no parquet", f"{len(uso_sem_parquet):,}",
       len(uso_sem_parquet), "Nenhum")

usuário-mês com treino no parquet: 150,572
  sem linha no uso_mensal: 2,173 (1,998 contas; 47 delas sem nenhuma linha em 2026)
  com linha no uso_mensal e atividades = 0: 12,702
usuário-mês com atividades > 0 no uso_mensal e nenhum treino no parquet: 0


## 2. Validade

### 2.1 Tipo e origem

O dicionário não lista os valores de `tipo`. O enunciado descreve o produto com seis: corrida, caminhada e pedal (ao ar
livre) e funcional, força e yoga (vídeo). A `origem` tem três valores no dicionário: `app`, `garmin` e `apple_health`.

In [8]:
TIPOS_ENUNCIADO = {"corrida", "caminhada", "pedal", "funcional", "forca", "yoga"}
tipo_fora = ~ativ["tipo"].isin(TIPOS_ENUNCIADO)
print("tipo:", ativ["tipo"].value_counts().to_dict())
print(f"fora dos seis do enunciado: {tipo_fora.sum():,} -> {ativ.loc[tipo_fora, 'tipo'].value_counts().to_dict()}")
print("origem desses:", ativ.loc[tipo_fora, "origem"].value_counts().to_dict(),
      "| primeiro:", ativ.loc[tipo_fora, "dh_br"].min())
achado("Validade", "tipo fora dos seis do enunciado",
       f"{tipo_fora.sum():,} linhas em inglês (run, walk, ride, strength, hiit), todas do app, a partir de 15/04/2026",
       tipo_fora.sum(),
       "Mapear para o português antes de qualquer contagem por tipo: run→corrida, walk→caminhada, ride→pedal, "
       "strength→forca, hiit→funcional (equivalência testada em 5.4)")

ORIGENS = {"app", "garmin", "apple_health"}
origem_fora = int((~ativ["origem"].isin(ORIGENS)).sum())
print("origem:", ativ["origem"].value_counts().to_dict(), "| fora do dicionário:", origem_fora)
achado("Validade", "origem fora da lista do dicionário", f"{origem_fora:,} linhas", origem_fora, "Nenhum")

tipo: {'corrida': 223818, 'run': 162108, 'funcional': 145476, 'caminhada': 118024, 'yoga': 113849, 'forca': 112704, 'hiit': 109934, 'walk': 88509, 'strength': 85548, 'pedal': 53740, 'ride': 40765}
fora dos seis do enunciado: 486,864 -> {'run': 162108, 'hiit': 109934, 'walk': 88509, 'strength': 85548, 'ride': 40765}
origem desses: {'app': 486864} | primeiro: 2026-04-15 05:00:00


origem: {'app': 881706, 'garmin': 207954, 'apple_health': 164815} | fora do dicionário: 0


### 2.2 Datas e identificadores

In [9]:
print("padrões de inicio:", ativ["inicio"].str.replace(r"\d", "9", regex=True).value_counts().to_dict())
data_invalida = int(ativ["inicio_txt"].isna().sum())
achado("Validade", "inicio que não converte em data", f"{data_invalida:,} linhas; um único formato, sem fuso",
       data_invalida, "Nenhum")

PADRAO_USER = r"u_[0-9a-f]{8}"
id_fora = ~ativ["atividade_id"].str.fullmatch(r"a_\d{7}")
user_fora = ~ativ["user_id"].str.fullmatch(PADRAO_USER)
com = ativ.loc[~vazio(ativ["treino_com"]), "treino_com"]
com_fora = ~com.str.fullmatch(PADRAO_USER)
sem_cadastro = ~ativ["user_id"].isin(usuarios["user_id"])
com_sem_cadastro = ~com.isin(usuarios["user_id"])
print(f"atividade_id fora do padrão a_0000000: {id_fora.sum():,} | user_id fora do padrão: {user_fora.sum():,} | "
      f"sem cadastro: {sem_cadastro.sum():,}")
print(f"treino_com fora do padrão: {com_fora.sum():,} | sem cadastro: {com_sem_cadastro.sum():,}")
achado("Validade", "identificador fora do padrão ou conta sem cadastro",
       f"{id_fora.sum():,} atividade_id; {user_fora.sum():,} user_id e {com_fora.sum():,} treino_com fora do padrão; "
       f"{sem_cadastro.sum() + com_sem_cadastro.sum():,} sem cadastro",
       (id_fora | user_fora | sem_cadastro).sum() + (com_fora | com_sem_cadastro).sum(), "Se houver: excluir e registrar")

padrões de inicio: {'9999-99-99 99:99:99': 1254475}


atividade_id fora do padrão a_0000000: 0 | user_id fora do padrão: 0 | sem cadastro: 0
treino_com fora do padrão: 0 | sem cadastro: 0


### 2.3 Números dentro do plausível

Faixas por tipo: duração acima de zero; ritmo de corrida entre 2,5 e 15 min/km, de caminhada entre 6 e 25 e de pedal
entre 1 e 8; frequência cardíaca média entre 40 e 220.

In [10]:
display(ativ.groupby("tipo_pt")["duracao_min"].describe(percentiles=[.5, .99, .999]).round(1))

dur_zero = ativ["duracao_min"] == 0
print(f"duração 0: {dur_zero.sum():,}; por origem {ativ.loc[dur_zero, 'origem'].value_counts().to_dict()}; "
      f"com distância {ativ.loc[dur_zero, 'distancia_km'].notna().sum():,}, com FC {ativ.loc[dur_zero, 'fc_media'].notna().sum():,}")
print("  por mês:", ativ.loc[dur_zero, "mes_br"].value_counts().sort_index().to_dict())

,count,mean,std,min,50%,99%,99.9%,max
tipo_pt,,,,,,,,
caminhada,206533.0,58.9,18.9,0.0,58.5,97.7,101.9,103.9
corrida,385926.0,43.0,19.8,0.0,39.5,106.5,232.3,333.2
forca,198252.0,47.4,10.4,0.0,47.5,64.7,65.0,65.0
funcional,255410.0,32.4,7.4,0.0,32.5,44.7,45.0,45.0
pedal,94505.0,80.2,49.4,0.0,68.7,258.4,464.1,543.9
yoga,113849.0,37.3,10.3,0.0,37.3,54.6,55.0,55.0


duração 0: 3,649; por origem {'app': 2540, 'garmin': 607, 'apple_health': 502}; com distância 1,991, com FC 1,467
  por mês: {'2026-01': 449, '2026-02': 351, '2026-03': 436, '2026-04': 414, '2026-05': 436, '2026-06': 373, '2026-07': 348, '2026-08': 380, '2026-09': 462}


Duração 0 pode ser duas coisas: a gravação da duração falhou num treino que aconteceu, ou a pessoa abriu o treino e saiu
logo. A distância (maior que zero) e a frequência cardíaca são outra prova de que o treino aconteceu. Se a falha é só da
duração, esses treinos devem se parecer com os demais em distância, FC e tipo, e estar espalhados por muitas contas.

In [11]:
z = ativ[dur_zero]
outra_prova = (z["distancia_km"] > 0) | z["fc_media"].notna()
print(f"duração 0: {len(z):,} | com distância > 0 ou FC: {outra_prova.sum():,} ({100 * outra_prova.mean():.0f}%) | "
      f"sem outra prova: {(~outra_prova).sum():,}")
print("  sem outra prova, por tipo:", z.loc[~outra_prova, "tipo_pt"].value_counts().to_dict(),
      "| origem:", z.loc[~outra_prova, "origem"].value_counts().to_dict())

normais = ativ[ativ["duracao_min"] > 0]
reconstruida = (z["distancia_km"] * z["ritmo_min_km"]).dropna()
comparacao_zero = pd.DataFrame({
    "duração 0": [z.loc[z["distancia_km"] > 0, "distancia_km"].median(), z["fc_media"].median(), reconstruida.median()],
    "demais treinos": [normais["distancia_km"].median(), normais["fc_media"].median(),
                       normais.loc[normais["ritmo_min_km"].notna(), "duracao_min"].median()],
}, index=["distância mediana (km)", "FC mediana", "duração mediana com GPS (min; na duração 0, distância × ritmo)"])
display(comparacao_zero.round(1))
display(pd.DataFrame({"duração 0 (%)": (100 * z["tipo_pt"].value_counts(normalize=True)).round(1),
                      "todos (%)": (100 * ativ["tipo_pt"].value_counts(normalize=True)).round(1)}))
por_conta = z["user_id"].value_counts()
print(f"contas: {len(por_conta):,} | com um só caso: {(por_conta == 1).sum():,} | máximo por conta: {por_conta.max()}")

duração 0: 3,649 | com distância > 0 ou FC: 2,651 (73%) | sem outra prova: 998
  sem outra prova, por tipo: {'funcional': 435, 'forca': 328, 'yoga': 220, 'corrida': 9, 'caminhada': 4, 'pedal': 2} | origem: {'app': 998}


,duração 0,demais treinos
distância mediana (km),6.5,6.5
FC mediana,135.0,133.0
"duração mediana com GPS (min; na duração 0, distância × ritmo)",47.2,46.9


,duração 0 (%),todos (%)
tipo_pt,,
corrida,31.6,30.8
funcional,20.6,20.4
caminhada,15.9,16.5
forca,15.0,15.8
yoga,9.4,9.1
pedal,7.5,7.5


contas: 3,242 | com um só caso: 2,879 | máximo por conta: 4


Os treinos com distância ou FC se parecem com os demais: a falha é só da duração. Os que não têm nenhuma outra prova são
quase todos de vídeo, registrados no app. Para eles não dá para separar falha de gravação de vídeo aberto e fechado logo
em seguida, e por isso são marcados.

In [12]:
achado("Validade", "treino com duração 0",
       f"{dur_zero.sum():,} linhas, espalhadas por todos os meses, tipos, origens e {len(por_conta):,} contas; "
       f"{outra_prova.sum():,} têm distância ou FC e se parecem com os demais treinos; {(~outra_prova).sum():,} "
       "(quase todos de vídeo, do app) não têm outra prova",
       dur_zero.sum(),
       "Contar o treino e tratar a duração como ausente (fora de somas e médias de minutos). Marcar os que não têm "
       "distância nem FC, para rodar o modelo com e sem eles")

longo = ativ["duracao_min"] > 300
print(f"duração acima de 5h: {longo.sum():,} -> {ativ.loc[longo, 'tipo_pt'].value_counts().to_dict()}; "
      f"máximo {ativ['duracao_min'].max():.1f} min")
achado("Validade", "treino com mais de 5h",
       f"{longo.sum():,} linhas: {ativ.loc[longo, 'tipo_pt'].value_counts().to_dict()}; até {ativ['duracao_min'].max():.0f} min",
       longo.sum(), "Manter: pedal longo e corridas de ultra são plausíveis. Usar mediana, não média, ao resumir minutos",
       status="esperado")

FAIXA_RITMO = {"corrida": (2.5, 15), "caminhada": (6, 25), "pedal": (1, 8)}
lim = ativ["tipo_pt"].map(FAIXA_RITMO)
com_ritmo = ativ["ritmo_min_km"].notna() & lim.notna()
ritmo_fora = com_ritmo & ((ativ["ritmo_min_km"] < lim.where(com_ritmo).str[0]) | (ativ["ritmo_min_km"] > lim.where(com_ritmo).str[1]))
fc_fora = (ativ["fc_media"] < 40) | (ativ["fc_media"] > 220)
print(f"ritmo fora da faixa do tipo: {ritmo_fora.sum():,} | FC fora de 40-220: {fc_fora.sum():,}")
achado("Validade", "ritmo fora da faixa do tipo", f"{ritmo_fora.sum():,} linhas", ritmo_fora.sum(), "Nenhum")
achado("Validade", "frequência cardíaca fora de 40-220", f"{fc_fora.sum():,} linhas", fc_fora.sum(), "Nenhum")

duração acima de 5h: 607 -> {'pedal': 576, 'corrida': 31}; máximo 543.9 min


ritmo fora da faixa do tipo: 0 | FC fora de 40-220: 0


## 3. Consistência

### 3.1 Horário de início × origem: o Garmin grava em UTC?

Quem treina tem hábito de horário, e esse hábito não depende do relógio que a pessoa usa. Então a distribuição dos
treinos por hora do dia deveria ser parecida nas três origens. Se a do Garmin estiver deslocada 3h para frente, ele grava
em UTC (Brasília é UTC−3, sem horário de verão em 2026).

A medida é a distância entre as distribuições (metade da soma das diferenças em pontos percentuais): 0 quando são iguais,
100 quando não têm nenhuma hora em comum.

In [13]:
def dist_por_hora(horas):
    return horas.value_counts(normalize=True).reindex(range(24), fill_value=0)

app_h = dist_por_hora(ativ.loc[ativ["origem"] == "app", "inicio_txt"].dt.hour)
apple_h = dist_por_hora(ativ.loc[ativ["origem"] == "apple_health", "inicio_txt"].dt.hour)
garmin_txt = dist_por_hora(ativ.loc[eh_garmin, "inicio_txt"].dt.hour)
garmin_br = dist_por_hora(ativ.loc[eh_garmin, "dh_br"].dt.hour)
horas = pd.DataFrame({"app": app_h, "apple_health": apple_h, "garmin (texto)": garmin_txt, "garmin (−3h)": garmin_br})
display((100 * horas).round(1).T)

distancia = lambda a, b: round(50 * (a - b).abs().sum(), 1)
print(f"distância app x apple_health: {distancia(app_h, apple_h)} | app x garmin como está: {distancia(app_h, garmin_txt)} "
      f"| app x garmin com −3h: {distancia(app_h, garmin_br)}")
achado("Consistência", "horário do Garmin em UTC sem o Z",
       f"todos os {eh_garmin.sum():,} treinos do Garmin: a distribuição por hora fica igual à do app só com −3h "
       f"(distância {distancia(app_h, garmin_txt)} como está; {distancia(app_h, garmin_br)} com a conversão)",
       eh_garmin.sum(),
       "Converter o inicio do Garmin de UTC para Brasília (−3h) antes de qualquer conta por hora, dia ou mês; "
       "app e apple_health já estão em Brasília. Confirmado também pelos treinos a 3h exatas (4.3)")

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23
app,0.0,0.0,0.0,0.0,0.0,4.7,16.7,16.3,11.5,7.1,0.0,2.3,2.3,2.3,0.0,0.0,0.0,4.9,9.4,12.3,7.4,2.9,0.0,0.0
apple_health,0.0,0.0,0.0,0.0,0.0,4.9,16.9,16.4,11.4,7.0,0.0,2.4,2.3,2.3,0.0,0.0,0.0,4.8,9.6,12.1,7.3,2.6,0.0,0.0
garmin (texto),2.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,4.3,16.3,15.9,11.7,7.1,0.0,2.6,2.6,2.6,0.0,0.0,0.0,4.7,9.5,12.3,7.6
garmin (−3h),0.0,0.0,0.0,0.0,0.0,4.3,16.3,15.9,11.7,7.1,0.0,2.6,2.6,2.6,0.0,0.0,0.0,4.7,9.5,12.3,7.6,2.8,0.0,0.0


distância app x apple_health: 0.8 | app x garmin como está: 76.4 | app x garmin com −3h: 1.5


### 3.2 Origem × relógio conectado à conta

Treino do Garmin só deveria vir de conta com Garmin; do Apple Health, só de conta com Apple Watch.

In [14]:
display(pd.crosstab(ativ["origem"], ativ["relogio"]))
origem_sem_relogio = int(((eh_garmin & (ativ["relogio"] != "garmin")) |
                          ((ativ["origem"] == "apple_health") & (ativ["relogio"] != "apple_watch"))).sum())
achado("Consistência", "treino de relógio em conta sem aquele relógio", f"{origem_sem_relogio:,} linhas",
       origem_sem_relogio, "Nenhum")

relogio,apple_watch,garmin,nenhum
origem,,,
app,74753,41966,764987
apple_health,164815,0,0
garmin,0,207954,0


### 3.3 Frequência cardíaca × relógio

O dicionário diz que a FC vem "quando há relógio". Há FC em treinos com origem `app`: a hipótese é que o app lê a FC do
relógio conectado à conta mesmo quando o treino é registrado no app. Se for isso, FC preenchida deve coincidir
exatamente com conta que tem relógio.

In [15]:
tem_fc = ativ["fc_media"].notna()
tem_relogio = ativ["relogio"] != "nenhum"
display(pd.crosstab([ativ["origem"], ativ["relogio"]], tem_fc.rename("tem FC")))
fc_sem_relogio = int((tem_fc & ~tem_relogio).sum())
relogio_sem_fc = int((~tem_fc & tem_relogio).sum())
fc_no_app = int((tem_fc & (ativ["origem"] == "app")).sum())
print(f"FC sem relógio na conta: {fc_sem_relogio:,} | conta com relógio e treino sem FC: {relogio_sem_fc:,} | "
      f"FC em treino com origem app: {fc_no_app:,}")
achado("Consistência", "FC preenchida × relógio conectado à conta", f"{fc_sem_relogio + relogio_sem_fc:,} divergências",
       fc_sem_relogio + relogio_sem_fc, "Nenhum")
achado("Consistência", "FC em treino registrado pelo app",
       f"{fc_no_app:,} linhas, todas de contas com relógio: o app lê a FC do relógio conectado",
       fc_no_app, "Não é erro. Para saber se há FC, usar relogio_conectado da conta, não a origem do treino",
       status="esperado")

tem FC                     False   True 
origem       relogio                    
app          apple_watch       0   74753
             garmin            0   41966
             nenhum       764987       0
apple_health apple_watch       0  164815
garmin       garmin            0  207954

FC sem relógio na conta: 0 | conta com relógio e treino sem FC: 0 | FC em treino com origem app: 116,719


### 3.4 Ritmo × duração ÷ distância

O ritmo (min/km) deveria ser a duração dividida pela distância. Diferença acima de 0,1 min/km é divergência.

In [16]:
r = ativ[ativ["distancia_km"].notna() & ativ["ritmo_min_km"].notna()]
calc = r["duracao_min"] / r["distancia_km"]
diverge = (calc - r["ritmo_min_km"]).abs() > 0.1
dur0 = r["duracao_min"] == 0
dist_esperada = r["duracao_min"] / r["ritmo_min_km"]
dist_absurda = ~dur0 & (r["distancia_km"] > 2 * dist_esperada)
print(f"treinos com distância e ritmo: {len(r):,} | divergentes: {diverge.sum():,}")
print(f"  duração 0 (já em 2.3): {(diverge & dur0).sum():,} | distância mais que o dobro de duração ÷ ritmo: "
      f"{(diverge & dist_absurda).sum():,} | outros: {(diverge & ~dur0 & ~dist_absurda).sum():,}")
print("  distâncias absurdas (km):", r.loc[dist_absurda, "distancia_km"].describe().round(1).to_dict())
print("  razão distância / (duração ÷ ritmo) nos demais:",
      (r.loc[~dur0 & ~dist_absurda, "distancia_km"] / dist_esperada[~dur0 & ~dist_absurda]).quantile([.001, .5, .999]).round(3).to_dict())
r.loc[dist_absurda, ["tipo", "origem", "duracao_min", "distancia_km", "ritmo_min_km"]].head()

treinos com distância e ritmo: 672,185 | divergentes: 2,971
  duração 0 (já em 2.3): 1,972 | distância mais que o dobro de duração ÷ ritmo: 999 | outros: 0
  distâncias absurdas (km): {'count': 999.0, 'mean': 995.6, 'std': 1187.1, 'min': 104.7, '25%': 405.2, '50%': 628.7, '75%': 1035.8, 'max': 10012.8}
  razão distância / (duração ÷ ritmo) nos demais: {0.001: 0.996, 0.5: 1.0, 0.999: 1.004}


,tipo,origem,duracao_min,distancia_km,ritmo_min_km
1192,ride,app,30.5,833.79,2.50
2141,corrida,garmin,25.2,668.30,4.50
2422,caminhada,apple_health,55.2,381.67,11.18
4218,corrida,app,47.5,756.63,7.21
4646,walk,app,37.4,169.21,11.71


In [17]:
achado("Consistência", "distância incoerente com duração e ritmo",
       f"{dist_absurda.sum():,} treinos com distância de "
       f"{r.loc[dist_absurda, 'distancia_km'].min():.0f} a {r.loc[dist_absurda, 'distancia_km'].max():,.0f} km; "
       "o ritmo bate com o tipo e a duração é plausível, só a distância está errada",
       dist_absurda.sum(),
       "Tratar a distância como erro e recalcular: distância = duração ÷ ritmo. Nos outros "
       f"{(~dur0 & ~dist_absurda).sum():,} treinos as três colunas batem")

### 3.5 Treino × cadastro da conta e companhia

In [18]:
antes_hora = ativ["dh_br"] < ativ["cadastro"]
antes_dia = ativ["data_br"] < ativ["cadastro"].dt.normalize()
print(f"treino antes do cadastro: {antes_hora.sum():,}; num dia anterior ao do cadastro: {antes_dia.sum():,}")
print("  horas entre o treino e o cadastro, no mesmo dia:",
      ((ativ.loc[antes_hora, "cadastro"] - ativ.loc[antes_hora, "dh_br"]).dt.total_seconds() / 3600).describe().round(1).to_dict())
print("  origem:", ativ.loc[antes_hora, "origem"].value_counts().to_dict())
achado("Consistência", "treino anterior ao cadastro da conta",
       f"{antes_hora.sum():,} treinos, todos no próprio dia do cadastro, algumas horas antes; nenhum em dia anterior",
       antes_hora.sum(),
       "Comparar treino e cadastro por dia, não por hora. Hipótese a confirmar com o dono do dado: a hora do cadastro "
       "vem de outro sistema (marketing) e não é a hora em que a conta foi criada no app")

com_si = int((ativ["treino_com"] == ativ["user_id"]).sum())
tem_com = ~vazio(ativ["treino_com"])
print(f"treinos com companhia: {tem_com.sum():,} ({ativ.loc[tem_com, 'treino_com'].nunique():,} contas marcadas) | "
      f"companhia igual ao próprio usuário: {com_si:,}")
achado("Consistência", "treino_com igual ao próprio usuário", f"{com_si:,} linhas", com_si, "Nenhum")

treino antes do cadastro: 5,069; num dia anterior ao do cadastro: 0
  horas entre o treino e o cadastro, no mesmo dia: {'count': 5069.0, 'mean': 6.9, 'std': 4.7, 'min': 0.0, '25%': 2.7, '50%': 6.4, '75%': 10.9, 'max': 17.9}
  origem: {'app': 3585, 'garmin': 891, 'apple_health': 593}


treinos com companhia: 14,137 (2,765 contas marcadas) | companhia igual ao próprio usuário: 0


### 3.6 Quantidade por usuário e mês × `uso_mensal.csv`

O `uso_mensal.csv` diz consolidar "quantos treinos foram registrados" por usuário e mês. Comparamos com a contagem do
parquet considerando todas as origens e só algumas. A hipótese, vinda do primeiro olhar, é que o BI conta só os treinos
registrados no app e deixa de fora os sincronizados do relógio.

In [19]:
comparacao = []
contagens = {}
for nome, filtro in [("todas as origens", ativ["origem"].isin(ORIGENS)), ("só app", ativ["origem"] == "app"),
                     ("app + garmin", ativ["origem"].isin(["app", "garmin"])),
                     ("app + apple_health", ativ["origem"].isin(["app", "apple_health"]))]:
    n = ativ[filtro].groupby(["user_id", "mes_br"]).size().rename("n")
    n.index.names = ["user_id", "mes"]
    c = uso26[["atividades"]].join(n, how="outer").fillna(0)
    contagens[nome] = c
    comparacao.append({"parquet": nome, "treinos no parquet": int(c["n"].sum()), "treinos no uso_mensal": int(c["atividades"].sum()),
                       "% usuário-mês iguais": round(100 * (c["atividades"] == c["n"]).mean(), 1)})
display(pd.DataFrame(comparacao))

c = contagens["só app"]
dif = (c["atividades"] - c["n"]).astype(int)
print("uso_mensal − treinos do app, por usuário-mês:", dif.value_counts().head(9).to_dict())

relogio_conta = info["relogio_conectado"]
c_all = contagens["todas as origens"].reset_index()
c_all["relogio"] = c_all["user_id"].map(relogio_conta)
captura = c_all.groupby("relogio")[["atividades", "n"]].sum()
captura["% capturado pelo uso_mensal"] = (100 * captura["atividades"] / captura["n"]).round(1)
display(captura)

,parquet,treinos no parquet,treinos no uso_mensal,% usuário-mês iguais
0,todas as origens,1254475,867831,86.2
1,só app,881706,867831,91.5
2,app + garmin,1089660,867831,88.4
3,app + apple_health,1046521,867831,89.3


uso_mensal − treinos do app, por usuário-mês: {0: 402662, -1: 15862, 1: 11033, -2: 4401, 2: 2919, -3: 1615, 3: 613, -4: 561, -5: 188}


,atividades,n,% capturado pelo uso_mensal
relogio,,,
apple_watch,70371.0,239568.0,29.4
garmin,35638.0,249920.0,14.3
nenhum,761822.0,764987.0,99.6


In [20]:
fora_uso = int((ativ["origem"] != "app").sum())
achado("Consistência", "uso_mensal.atividades × contagem do parquet",
       f"uso_mensal soma {int(c['atividades'].sum()):,} treinos em 2026 contra {N:,} no parquet; bate com os treinos do app "
       f"em {round(100 * (dif == 0).mean(), 1)}% dos usuário-mês e deixa de fora os {fora_uso:,} sincronizados do relógio. "
       f"Para contas com relógio, captura {captura.loc['garmin', '% capturado pelo uso_mensal']}% (Garmin) e "
       f"{captura.loc['apple_watch', '% capturado pelo uso_mensal']}% (Apple Watch) dos treinos",
       fora_uso,
       "Em 2026, medir uso pelo parquet. Para comparar 2025 (só existe no uso_mensal) com 2026, usar só a origem app "
       "nos dois anos. A diferença de ±1 a ±5 treinos que sobra em ~8% dos usuário-mês fica em aberto com o dono do dado")

## 4. Unicidade

### 4.1 Identificador e linha repetida

In [21]:
dup_id = int(ativ["atividade_id"].duplicated().sum())
dup_linha = int(ativ.duplicated(subset=COLUNAS).sum())
sem_id = ativ.duplicated(subset=[c for c in COLUNAS if c != "atividade_id"], keep=False)
print(f"atividade_id repetido: {dup_id:,} | linha inteira repetida: {dup_linha:,} | "
      f"mesmo treino com atividade_id diferente: {int(sem_id.sum()) // 2:,} par(es)")
display(ativ.loc[sem_id, COLUNAS])
achado("Unicidade", "atividade_id repetido ou linha inteira repetida", f"{dup_id:,} e {dup_linha:,}", dup_id + dup_linha, "Nenhum")
achado("Unicidade", "mesmo treino com atividade_id diferente", f"{int(sem_id.sum()) // 2:,} cópia(s)", int(sem_id.sum()) // 2,
       "Remover a cópia, mantendo o menor atividade_id")

atividade_id repetido: 0 | linha inteira repetida: 0 | mesmo treino com atividade_id diferente: 1 par(es)


,atividade_id,user_id,inicio,tipo,duracao_min,distancia_km,ritmo_min_km,fc_media,origem,gps_ok,treino_com
1013677,a_0781469,u_cea15907,2026-05-14 07:59:00,strength,60.4,NaN,NaN,NaN,app,True,
1013678,a_0861861,u_cea15907,2026-05-14 07:59:00,strength,60.4,NaN,NaN,NaN,app,True,


### 4.2 Dois treinos do mesmo usuário no mesmo minuto

Quem registra no app e tem relógio conectado pode ter o mesmo treino gravado duas vezes: uma pelo app e outra pela
sincronização. Se for isso, os pares de origens diferentes têm o mesmo tipo e a mesma duração.

In [22]:
CHAVE = ["user_id", "dh_br"]
mesmo_min = ativ[ativ.duplicated(CHAVE, keep=False)]
grupos = mesmo_min.groupby(CHAVE).agg(
    treinos=("atividade_id", "size"), origens=("origem", "nunique"), tipos=("tipo_pt", "nunique"),
    dif_duracao=("duracao_min", lambda s: s.max() - s.min()),
    combinacao=("origem", lambda s: " + ".join(sorted(s))))
grupos["classe"] = "mesma origem, tipos diferentes"
grupos.loc[(grupos["origens"] == 1) & (grupos["tipos"] == 1), "classe"] = "mesma origem, mesmo tipo"
grupos.loc[(grupos["origens"] > 1) & (grupos["tipos"] == 1), "classe"] = "app + relógio, mesmo tipo"
grupos.loc[(grupos["origens"] > 1) & (grupos["tipos"] > 1), "classe"] = "app + relógio, tipos diferentes"
grupos["sobra"] = grupos["treinos"] - 1
resumo_min = grupos.groupby("classe").agg(grupos=("treinos", "size"), linhas_a_mais=("sobra", "sum"),
                                          duracao_igual=("dif_duracao", lambda s: int((s < 0.05).sum())),
                                          dif_duracao_mediana=("dif_duracao", "median"))
display(resumo_min)
display(grupos["combinacao"].value_counts().to_frame("grupos"))

,grupos,linhas_a_mais,duracao_igual,dif_duracao_mediana
classe,,,,
"app + relógio, mesmo tipo",1984,1990,1891,0.00
"app + relógio, tipos diferentes",116,118,0,18.75
"mesma origem, mesmo tipo",579,579,2,15.00
"mesma origem, tipos diferentes",680,680,0,15.50


,grupos
combinacao,
app + garmin,1103
app + apple_health,989
app + app,905
garmin + garmin,197
apple_health + apple_health,157
app + garmin + garmin,5
app + apple_health + apple_health,2
app + app + garmin,1


Os pares app + relógio do mesmo tipo confirmam a hipótese de gravação dupla: quase todos têm a mesma duração. Os demais
grupos não se explicam assim: são dois treinos no mesmo minuto com tipo ou duração diferentes (yoga e caminhada, por
exemplo), e só 2 têm a mesma duração. Parecem treinos distintos que começaram no mesmo minuto. Fica como hipótese a
confirmar com o dono do dado.

In [23]:
def classe(nome):
    return resumo_min.loc[nome] if nome in resumo_min.index else pd.Series({"grupos": 0, "linhas_a_mais": 0, "duracao_igual": 0})

dupla = classe("app + relógio, mesmo tipo")
achado("Unicidade", "mesmo treino gravado pelo app e pelo relógio",
       f"{int(dupla['grupos']):,} pares no mesmo minuto, mesmo tipo, origens diferentes; "
       f"{int(dupla['duracao_igual']):,} com a mesma duração",
       dupla["linhas_a_mais"],
       "Remover a cópia do app e manter a do relógio, antes de contar treinos ou somar minutos")

outros = [classe("mesma origem, mesmo tipo"), classe("mesma origem, tipos diferentes"), classe("app + relógio, tipos diferentes")]
achado("Unicidade", "dois treinos diferentes do mesmo usuário no mesmo minuto",
       f"{int(sum(o['grupos'] for o in outros)):,} grupos: {int(outros[0]['grupos']):,} da mesma origem e mesmo tipo "
       f"(só {int(outros[0]['duracao_igual']):,} com a mesma duração), {int(outros[1]['grupos']):,} da mesma origem com tipos diferentes e "
       f"{int(outros[2]['grupos']):,} de app e relógio com tipos diferentes",
       sum(o["linhas_a_mais"] for o in outros),
       "Hipótese a confirmar com o dono do dado. Até lá, manter os dois e marcar; o efeito na contagem é pequeno")

### 4.3 Treinos a exatamente 3h de distância

Se o Garmin grava em UTC, os pares app + Garmin do mesmo treino aparecem a exatamente 3h de distância lendo o texto e no
mesmo minuto depois da conversão.

In [24]:
def a_tres_horas(col):
    s = ativ.sort_values(["user_id", col])
    mesmo = s["user_id"] == s["user_id"].shift()
    tres = mesmo & ((s[col] - s[col].shift()).dt.total_seconds() == 3 * 3600)
    par = s["origem"].shift() + " + " + s["origem"]
    return int(tres.sum()), par[tres].value_counts().to_dict()

n_txt, par_txt = a_tres_horas("inicio_txt")
n_br, par_br = a_tres_horas("dh_br")
print(f"pares a 3h exatas lendo o texto: {n_txt:,} -> {par_txt}")
print(f"pares a 3h exatas em horário de Brasília: {n_br:,} -> {par_br}")
app_garmin_br = sum(v for k, v in par_br.items() if "garmin" in k and "app" in k)
app_garmin_txt = sum(v for k, v in par_txt.items() if "garmin" in k and "app" in k)
mesmo_min_txt = int(ativ.groupby(["user_id", "inicio_txt"])["origem"]
                    .agg(lambda s: {"app", "garmin"} <= set(s)).sum())
print(f"pares app + Garmin no mesmo minuto lendo o texto (ficam a 3h depois da conversão): {mesmo_min_txt:,}")

pares a 3h exatas lendo o texto: 1,005 -> {'app + garmin': 901, 'app + app': 72, 'garmin + garmin': 14, 'apple_health + app': 7, 'app + apple_health': 6, 'apple_health + apple_health': 5}
pares a 3h exatas em horário de Brasília: 109 -> {'app + app': 72, 'garmin + garmin': 12, 'apple_health + app': 7, 'app + apple_health': 6, 'apple_health + apple_health': 5, 'garmin + app': 4, 'app + garmin': 3}


pares app + Garmin no mesmo minuto lendo o texto (ficam a 3h depois da conversão): 5


Lendo o texto, os pares de app e Garmin a 3h exatas são quase todos a gravação dupla da seção 4.2. Com a conversão, eles
caem no mesmo minuto. Dos poucos pares de app e Garmin que ficam a 3h depois da conversão, a maioria estava no mesmo
minuto no texto: treinos diferentes que, sem a conversão, pareceriam simultâneos.

In [25]:
achado("Unicidade", "treinos do mesmo usuário a exatamente 3h",
       f"{n_txt:,} pares lendo o texto ({app_garmin_txt:,} de app e Garmin); {n_br:,} depois da conversão "
       f"({app_garmin_br:,} de app e Garmin; {mesmo_min_txt:,} deles estavam no mesmo minuto no texto)",
       n_br, "Nenhum além da conversão do Garmin (3.1): o que sobra são treinos distintos, não um registro em dois fusos",
       status="esperado")

## 5. Temporalidade

### 5.1 Período coberto e efeito do fuso

O dicionário diz que o arquivo vai de 01/01 a 30/09/2026.

In [26]:
print(f"primeiro treino: {ativ['dh_br'].min()} | último: {ativ['dh_br'].max()} (horário de Brasília)")
antes = int((ativ["dh_br"] < INICIO_ARQUIVO).sum())
depois = int((ativ["dh_br"] >= FIM_CORTE).sum())
depois_texto = int((ativ["inicio_txt"] >= FIM_CORTE).sum())
vira_dia = int((ativ["data_br"] != ativ["inicio_txt"].dt.normalize()).sum())
vira_mes = int((ativ["mes_br"] != ativ["inicio_txt"].dt.strftime("%Y-%m")).sum())
print(f"antes de 01/01/2026: {antes:,} | depois do corte: {depois:,} | 'depois do corte' lendo só o texto: {depois_texto:,}")
print(f"mudam de dia com a conversão: {vira_dia:,} | mudam de mês: {vira_mes:,}")
achado("Temporalidade", "treinos fora do período 01/01 a 30/09/2026", f"{antes:,} antes e {depois:,} depois", antes + depois,
       "Nenhum")
achado("Temporalidade", "treinos que mudam de dia ao acertar o fuso do Garmin",
       f"{vira_dia:,} mudam de dia e {vira_mes:,} de mês; lendo só o texto, {depois_texto:,} pareceriam posteriores ao corte",
       vira_dia, "Usar a data convertida (dh_br) para contar por dia ou mês")

primeiro treino: 2026-01-01 05:00:00 | último: 2026-09-30 21:59:00 (horário de Brasília)


antes de 01/01/2026: 0 | depois do corte: 0 | 'depois do corte' lendo só o texto: 34
mudam de dia com a conversão: 5,828 | mudam de mês: 188


### 5.2 Volume por mês

In [27]:
volume = pd.crosstab(ativ["mes_br"], ativ["origem"], margins=True, margins_name="total").drop(index="total")
volume["variação"] = volume["total"].pct_change().round(3)
display(volume)
meses_faltando = pd.period_range("2026-01", "2026-09", freq="M").astype(str).difference(volume.index)
saltos = int((volume["variação"].abs() > 0.5).sum())
print(f"meses sem treino: {list(meses_faltando)} | variação acima de 50%: {saltos}")
achado("Temporalidade", "meses sem treino ou com salto acima de 50%", f"{len(meses_faltando)} meses vazios; {saltos} saltos",
       len(meses_faltando) + saltos, "Nenhum")

origem,app,apple_health,garmin,total,variação
mes_br,,,,,
2026-01,110675,20209,26820,157704,NaN
2026-02,84935,15634,20619,121188,-0.232
2026-03,97366,18025,24068,139459,0.151
2026-04,100561,18315,24445,143321,0.028
2026-05,101801,19354,24868,146023,0.019
2026-06,88884,16863,20779,126526,-0.134
2026-07,89222,16881,19950,126053,-0.004
2026-08,101356,19157,22507,143020,0.135
2026-09,106906,20377,23898,151181,0.057


meses sem treino: [] | variação acima de 50%: 0


### 5.3 A falha de GPS de maio e junho

Os treinos ao ar livre sem distância ou sem ritmo (seção 1.2) têm todos `gps_ok = False`. A hipótese é uma falha do app
num período: se for, `gps_ok = False` aparece só no app e só nesse intervalo.

In [28]:
display(pd.crosstab(ativ["mes_br"], ativ["gps_ok"].map({True: "gps ok", False: "gps falhou"})))
falhou = ~ativ["gps_ok"]
print(f"gps_ok = False: {falhou.sum():,}; primeiro {ativ.loc[falhou, 'dh_br'].min()}, último {ativ.loc[falhou, 'dh_br'].max()}")
print("origem:", ativ.loc[falhou, "origem"].value_counts().to_dict())
janela = (ativ["dh_br"] >= ativ.loc[falhou, "dh_br"].min()) & (ativ["dh_br"] <= ativ.loc[falhou, "dh_br"].max())
ar_app_janela = janela & ativ["ar_livre"] & (ativ["origem"] == "app")
print(f"treinos ao ar livre do app na janela: {ar_app_janela.sum():,}; com GPS falho: {(ar_app_janela & falhou).sum():,} "
      f"({100 * (ar_app_janela & falhou).sum() / ar_app_janela.sum():.1f}%)")
print(f"contas afetadas: {ativ.loc[falhou, 'user_id'].nunique():,}")

gps_ok,gps falhou,gps ok
mes_br,,
2026-01,0,157704
2026-02,0,121188
2026-03,0,139459
2026-04,0,143321
2026-05,8078,137945
2026-06,6701,119825
2026-07,0,126053
2026-08,0,143020
2026-09,0,151181


gps_ok = False: 14,779; primeiro 2026-05-04 05:10:00, último 2026-06-28 09:59:00
origem: {'app': 14779}
treinos ao ar livre do app na janela: 96,019; com GPS falho: 14,779 (15.4%)
contas afetadas: 4,320


O achado está registrado em Completude (1.2). Para o churn, vale olhar se quem teve GPS falho nessa janela saiu mais
(hipótese a testar com `eventos_assinatura.csv`, e com `tickets_suporte.csv`, que tem chamados como "corrida registrada com
distância zerada").

### 5.4 Os tipos em inglês a partir de 15/04/2026

Se `run`, `walk`, `ride`, `strength` e `hiit` são só novos nomes de corrida, caminhada, pedal, força e funcional, então:
(a) os nomes em português somem do app quando os em inglês aparecem; (b) cada par tem a mesma distribuição de duração,
distância, ritmo e FC.

In [29]:
app = ativ[ativ["origem"] == "app"]
display(pd.crosstab(app["mes_br"], app["tipo"]))
print("primeira data de cada nome em inglês:", ativ[ativ["tipo"].isin(TIPO_PT)].groupby("tipo")["dh_br"].min().dt.date.to_dict())

pos = app[app["dh_br"] >= "2026-04-15"]
resid = pos[pos["tipo"].isin(TIPO_PT.values())]
print(f"nomes em português no app depois de 15/04: {len(resid):,} de {len(pos):,}; "
      f"{100 * resid['fc_media'].notna().mean():.0f}% deles de contas com relógio")

pares = []
for en, pt in TIPO_PT.items():
    for col in ["duracao_min", "distancia_km", "ritmo_min_km", "fc_media"]:
        a, b = ativ.loc[ativ["tipo"] == pt, col].dropna(), ativ.loc[ativ["tipo"] == en, col].dropna()
        if len(a) and len(b):
            pares.append({"par": f"{pt} / {en}", "coluna": col, "mediana pt": a.median(), "mediana en": b.median(),
                          "p90 pt": a.quantile(.9), "p90 en": b.quantile(.9)})
display(pd.DataFrame(pares).round(2))

tipo,caminhada,corrida,forca,funcional,hiit,pedal,ride,run,strength,walk,yoga
mes_br,,,,,,,,,,,
2026-01,17818,34961,17303,22370,0,8330,0,0,0,0,9893
2026-02,13632,27282,13141,16892,0,6443,0,0,0,0,7545
2026-03,15706,31154,15150,19481,0,7334,0,0,0,0,8541
2026-04,7600,15051,7442,9460,10543,3601,3992,16757,8300,8682,9133
2026-05,191,334,174,241,20308,71,7737,31413,15719,16568,9045
2026-06,172,291,141,171,18089,71,6797,26554,13947,14559,8092
2026-07,148,309,140,213,18201,73,6871,26010,14354,14549,8354
2026-08,172,356,172,228,20818,70,7349,29953,16118,16770,9350
2026-09,184,321,170,237,21975,79,8019,31421,17110,17381,10009


primeira data de cada nome em inglês: {'hiit': datetime.date(2026, 4, 15), 'ride': datetime.date(2026, 4, 15), 'run': datetime.date(2026, 4, 15), 'strength': datetime.date(2026, 4, 15), 'walk': datetime.date(2026, 4, 15)}
nomes em português no app depois de 15/04: 5,252 de 541,796; 100% deles de contas com relógio


,par,coluna,mediana pt,mediana en,p90 pt,p90 en
0,corrida / run,duracao_min,39.40,39.50,65.80,66.00
1,corrida / run,distancia_km,6.51,6.42,11.87,11.84
2,corrida / run,ritmo_min_km,6.03,6.03,7.33,7.34
3,corrida / run,fc_media,152.00,152.00,165.00,165.00
4,caminhada / walk,duracao_min,58.40,58.50,84.50,84.80
5,caminhada / walk,distancia_km,5.25,5.17,7.45,7.45
6,caminhada / walk,ritmo_min_km,11.25,11.24,12.66,12.65
7,caminhada / walk,fc_media,112.00,112.00,123.00,123.00
8,pedal / ride,duracao_min,69.00,68.30,137.90,136.60
9,pedal / ride,distancia_km,25.91,25.33,50.54,49.94


O achado está registrado em Validade (2.1): os pares têm a mesma distribuição e os nomes em português somem do app em
15/04. Os poucos que sobram vêm de contas com relógio, e é provável que o nome venha do relógio, e não do app.
`hiit` não é um tipo novo: é o funcional com outro nome.

## 6. Resumo

In [30]:
resumo = pd.DataFrame(achados)
resumo["% do arquivo"] = (100 * resumo["linhas_afetadas"] / N).round(2)
resumo.to_csv(SAIDAS / "qualidade_atividades.csv", index=False)
print(resumo["status"].value_counts().to_dict())
resumo

{'ok': 15, 'atenção': 12, 'esperado': 3}


,criterio,checagem,status,resultado,linhas_afetadas,tratamento_sugerido,% do arquivo
0,Completude,vazio nas colunas que todo treino tem,ok,"0 células em atividade_id, user_id, inicio, ti...",0,Nenhum,0.00
1,Completude,treino de vídeo com distância ou ritmo,ok,0 linhas,0,Nenhum,0.00
2,Completude,treino ao ar livre sem distância ou sem ritmo,atenção,"7,324 sem distância e 7,455 com distância e se...",14779,Não imputar distância nem ritmo: marcar o trei...,1.18
3,Completude,treinos de contas de teste,ok,0 como autor e 0 como companhia,0,"Se houver: excluir, como em assinaturas.csv",0.00
4,Completude,usuário-mês com treino no parquet e sem linha ...,atenção,"2,173 usuário-mês de 1,998 contas; outros 12,7...",2173,"Para medir uso em 2026, partir do parquet, não...",0.17
5,Completude,usuário-mês com treino no uso_mensal e nenhum ...,ok,0,0,Nenhum,0.00
6,Validade,tipo fora dos seis do enunciado,atenção,"486,864 linhas em inglês (run, walk, ride, str...",486864,Mapear para o português antes de qualquer cont...,38.81
7,Validade,origem fora da lista do dicionário,ok,0 linhas,0,Nenhum,0.00
8,Validade,inicio que não converte em data,ok,"0 linhas; um único formato, sem fuso",0,Nenhum,0.00
9,Validade,identificador fora do padrão ou conta sem cada...,ok,0 atividade_id; 0 user_id e 0 treino_com fora ...,0,Se houver: excluir e registrar,0.00


In [31]:
hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados/brutos/ inalterado: True
